# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Load Datasets

In [2]:
trips = pd.read_csv(r"F:\ubar-trip-analysis-logistics\data\raw_data\Uber Trip Details.csv")
locations = pd.read_csv(r"F:\ubar-trip-analysis-logistics\data\raw_data\Location Table.csv")

# Inspect

In [3]:
trips.head(5)

,Trip ID,Pickup Time,Drop Off Time,passenger_count,trip_distance,PULocationID,DOLocationID,fare_amount,Surge Fee,Vehicle,Payment_type
0,1,6/1/2024 0:42,6/1/2024 1:04,1,5.60,79,226,19.5,2.0,UberX,Uber Pay
1,2,6/1/2024 0:06,6/1/2024 0:13,1,1.72,142,186,8.0,0.0,Uber Black,Cash
2,3,6/1/2024 0:08,6/1/2024 0:21,1,3.41,229,238,13.0,0.0,Uber Black,Cash
3,4,6/1/2024 0:28,6/1/2024 0:37,1,1.81,188,35,9.0,0.0,UberX,Cash
4,5,6/1/2024 0:38,6/1/2024 0:45,1,1.89,100,137,8.0,0.0,Uber Black,Cash


In [4]:
trips.shape

(103728, 11)

In [5]:
trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103728 entries, 0 to 103727
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Trip ID          103728 non-null  int64  
 1   Pickup Time      103728 non-null  object 
 2   Drop Off Time    103728 non-null  object 
 3   passenger_count  103728 non-null  int64  
 4   trip_distance    103728 non-null  float64
 5   PULocationID     103728 non-null  int64  
 6   DOLocationID     103728 non-null  int64  
 7   fare_amount      103728 non-null  float64
 8   Surge Fee        103728 non-null  float64
 9   Vehicle          103728 non-null  object 
 10  Payment_type     103728 non-null  object 
dtypes: float64(3), int64(4), object(4)
memory usage: 8.7+ MB


In [6]:
locations.head(5)

,LocationID,Location,City
0,1,Newark Airport,"Newark, New Jersey"
1,2,Jamaica Bay,Queens
2,3,Allerton/Pelham Gardens,The Bronx
3,4,Alphabet City,Manhattan
4,5,Arden Heights,Staten Island


In [7]:
locations.shape

(265, 3)

In [8]:
locations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   LocationID  265 non-null    int64 
 1   Location    264 non-null    object
 2   City        263 non-null    object
dtypes: int64(1), object(2)
memory usage: 6.3+ KB


# ETL: Cleaning & Transformation

# Clean column names

In [9]:
trips.columns = trips.columns.str.strip().str.lower().str.replace(" ", "_")

# Convert datetime

In [10]:
trips['pickup_time'] = pd.to_datetime(trips['pickup_time'])
trips['drop_off_time'] = pd.to_datetime(trips['drop_off_time'])

# Feature engineering

In [11]:
trips['trip_duration_minutes'] = (trips['drop_off_time'] - trips['pickup_time']).dt.total_seconds()/60
trips['total_fare'] = trips['fare_amount'] + trips['surge_fee']
trips['hour'] = trips['pickup_time'].dt.hour
trips['day'] = trips['pickup_time'].dt.day
trips['month'] = trips['pickup_time'].dt.month
trips['weekday'] = trips['pickup_time'].dt.day_name()

# Clean location table

In [12]:
locations.columns = locations.columns.str.strip().str.lower().str.replace(" ", "_")

# Merge pickup & dropoff locations

In [13]:
trips = trips.merge(locations, left_on="pulocationid", right_on="locationid", how="left", suffixes=("", "_pickup"))
trips = trips.merge(locations, left_on="dolocationid", right_on="locationid", how="left", suffixes=("", "_dropoff"))

In [14]:
trips.head(5)

,trip_id,pickup_time,drop_off_time,passenger_count,trip_distance,pulocationid,dolocationid,fare_amount,surge_fee,vehicle,...,hour,day,month,weekday,locationid,location,city,locationid_dropoff,location_dropoff,city_dropoff
0,1,2024-06-01 00:42:00,2024-06-01 01:04:00,1,5.60,79,226,19.5,2.0,UberX,...,0,1,6,Saturday,79,East Village,Brooklyn,226,Sunnyside,Queens
1,2,2024-06-01 00:06:00,2024-06-01 00:13:00,1,1.72,142,186,8.0,0.0,Uber Black,...,0,1,6,Saturday,142,Lincoln Square East,Manhattan,186,Penn Station/Madison Sq West,Manhattan
2,3,2024-06-01 00:08:00,2024-06-01 00:21:00,1,3.41,229,238,13.0,0.0,Uber Black,...,0,1,6,Saturday,229,Sutton Place/Turtle Bay North,Manhattan,238,Upper West Side North,Manhattan
3,4,2024-06-01 00:28:00,2024-06-01 00:37:00,1,1.81,188,35,9.0,0.0,UberX,...,0,1,6,Saturday,188,Prospect-Lefferts Gardens,Brooklyn,35,Brownsville,Brooklyn
4,5,2024-06-01 00:38:00,2024-06-01 00:45:00,1,1.89,100,137,8.0,0.0,Uber Black,...,0,1,6,Saturday,100,Garment District,Queens,137,Kips Bay,Manhattan
